# Phase 3 (resume): finish cross-dataset and KADID analysis

Before running: **Add Input → Your Work → Notebooks →** the first phase-3 notebook, so its saved checkpoints are reused.
Then **Save Version → Save & Run All (Commit)** with GPU and Internet on. About 45–75 minutes.

In [ ]:
%cd /kaggle/working
!rm -rf Learned-Image-Quality-Assessment-Benchmark
!git clone https://github.com/elise-love/Learned-Image-Quality-Assessment-Benchmark.git
%cd Learned-Image-Quality-Assessment-Benchmark
!pip install -q pyiqa timm

!mkdir -p /tmp/datasets && ln -sfn /tmp/datasets datasets
!python scripts/download_data.py --datasets koniq10k kadid10k livec spaq
!rm -f datasets/*.tgz && rm -rf datasets/SPAQ/TestImage

# Reuse the checkpoints saved by the first phase-3 run (attached via "Add Input").
# Any checkpoint that is not found is retrained with the same config.
!for r in vit_s16_seed0 resnet50_lr1e-4_seed0; do f=$(find /kaggle/input -name "$r.pth" 2>/dev/null | head -1); if [ -n "$f" ]; then mkdir -p checkpoints/$r && cp "$f" checkpoints/$r/best.pth && echo "reusing $f"; else echo "no saved checkpoint for $r, will retrain"; fi; done
!test -f checkpoints/vit_s16_seed0/best.pth || python -m src.train --config configs/vit_s16.yaml --seed 0 --force --set out_dir=results/phase3_runs results_table=results/phase3_runs.csv
!test -f checkpoints/resnet50_lr1e-4_seed0/best.pth || python -m src.train --config configs/resnet50_lr1e-4.yaml --seed 0 --force --set out_dir=results/phase3_runs results_table=results/phase3_runs.csv

# Pairs already finished in the first run are skipped automatically.
!python -m src.cross_eval --runs vit_s16_seed0 resnet50_lr1e-4_seed0
!python scripts/run_baselines.py --config configs/baselines_cross_resume.yaml
!python scripts/analyze_cross.py

!mkdir -p /kaggle/working/best_checkpoints && cp checkpoints/vit_s16_seed0/best.pth /kaggle/working/best_checkpoints/vit_s16_seed0.pth && cp checkpoints/resnet50_lr1e-4_seed0/best.pth /kaggle/working/best_checkpoints/resnet50_lr1e-4_seed0.pth
!rm -rf checkpoints datasets
!cd /kaggle/working && zip -qr results_phase3b.zip Learned-Image-Quality-Assessment-Benchmark/results
